In [ ]:
# =========================
# 1. Загрузка и парсинг XML
# =========================
import xml.etree.ElementTree as ET
import pandas as pd
import re

# Загружаем файл
tree = ET.parse("Эпикриз_1502_v1.xml")
root = tree.getroot()

# Пространство имен HL7 CDA
ns = {"hl7": "urn:hl7-org:v3"}

# Извлекаем текстовые поля (жалобы, диагнозы, анамнез и т.п.)
texts = []
for section in root.findall(".//hl7:section", ns):
    title = section.find("hl7:title", ns)
    content = section.find("hl7:text", ns)
    if title is not None and content is not None:
        texts.append({"title": title.text.strip(), "text": ET.tostring(content, encoding="unicode")})

df_texts = pd.DataFrame(texts)
df_texts.head()




In [11]:
# =========================
# Импорт библиотек
# =========================
import os
import re
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

import nltk
from nltk.corpus import stopwords

# =========================
# Настройки
# =========================
nltk.download("stopwords")
russian_stopwords = stopwords.words("russian")

ns = {"hl7": "urn:hl7-org:v3"}
input_dir = "epikriz2/epi"   # папка с xml-файлами

# =========================
# Функции
# =========================
def clean_text(text):
    text = re.sub(r"<[^>]*>", " ", text)  # убрать теги
    text = re.sub(r"[^А-Яа-я\s]", " ", text)  # оставить только буквы
    text = text.lower()
    return " ".join([w for w in text.split() if w not in russian_stopwords])

def parse_xml_file(filepath):
    tree = ET.parse(filepath)
    root = tree.getroot()
    texts = []
    for section in root.findall(".//hl7:section", ns):
        title = section.find("hl7:title", ns)
        content = section.find("hl7:text", ns)
        if title is not None and content is not None:
            texts.append({
                "file": os.path.basename(filepath),
                "title": title.text.strip() if title.text else "",
                "text": ET.tostring(content, encoding="unicode")
            })
    return texts

# =========================
# Чтение всех XML файлов
# =========================
all_texts = []
for fname in os.listdir(input_dir):
    if fname.endswith(".xml"):
        filepath = os.path.join(input_dir, fname)
        all_texts.extend(parse_xml_file(filepath))

df_texts = pd.DataFrame(all_texts)
df_texts["cleaned"] = df_texts["text"].apply(clean_text)
print("Количество документов:", len(df_texts))
display(df_texts.head())

# =========================
# Текст -> Число (дискретные методы)
# =========================
bow = CountVectorizer(max_features=500)
X_bow = bow.fit_transform(df_texts["cleaned"])

tfidf = TfidfVectorizer(max_features=500)
X_tfidf = tfidf.fit_transform(df_texts["cleaned"])

# =========================
# Текст -> Число (недискретные методы, sentence-transformers)
# =========================

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("distiluse-base-multilingual-cased-v2")
X_embed = model.encode(df_texts["cleaned"].tolist())

# =========================
# Пример числовых данных (витал-параметры)
# =========================
# В реальности их нужно парсить из XML, здесь — пример
vitals = {
    "temperature": 37.3,
    "bp_systolic": 140,
    "bp_diastolic": 90,
    "pulse": 81,
    "spo2": 94,
    "height": 152,
    "weight": 62
}
df_vitals = pd.DataFrame([vitals])

# Нормализация
scaler_minmax = MinMaxScaler()
scaler_std = StandardScaler()

df_vitals_minmax = scaler_minmax.fit_transform(df_vitals)
df_vitals_std = scaler_std.fit_transform(df_vitals)

print("MinMax нормализация:", df_vitals_minmax)
print("Standard нормализация:", df_vitals_std)

# =========================
# MLP классификатор (демонстрация)
# =========================
# создадим фиктивные метки (например, диагноз "0/1")
y = np.random.randint(0, 2, size=len(df_texts))

X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

mlp = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=300, random_state=42)
mlp.fit(X_train, y_train)

print("MLP Accuracy:", mlp.score(X_test, y_test))


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\PC2\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Количество документов: 26364


,file,title,text,cleaned
0,Эпикриз_100263_v1.xml,ДИАГНОЗЫ ПАЦИЕНТА,"<ns0:text xmlns:ns0=""urn:hl7-org:v3""> <ns...",заключительный клинический диагноз шифр тип те...
1,Эпикриз_100263_v1.xml,ЖАЛОБЫ ПРИ ПОСТУПЛЕНИИ,"<ns0:text xmlns:ns0=""urn:hl7-org:v3""> Сос...",состояние средней тяжести жалобы лихорадка тем...
2,Эпикриз_100263_v1.xml,СОЦИАЛЬНЫЙ АНАМНЕЗ,"<ns0:text xmlns:ns0=""urn:hl7-org:v3""> <ns...",занятость работающий место жительства город
3,Эпикриз_100263_v1.xml,Анамнез заболевания,"<ns0:text xmlns:ns0=""urn:hl7-org:v3""> В н...",ноябре г перенес коронавирусную инфекцию лабор...
4,Эпикриз_100263_v1.xml,ФИЗИКАЛЬНОЕ ОБСЛЕДОВАНИЕ,"<ns0:text xmlns:ns0=""urn:hl7-org:v3""> <ns...",средней степени тяжести ясное баллов обычной о...


MinMax нормализация: [[0. 0. 0. 0. 0. 0. 0.]]
Standard нормализация: [[0. 0. 0. 0. 0. 0. 0.]]
MLP Accuracy: 0.5069220557557368


In [13]:
import os
import re
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

import nltk
from nltk.corpus import stopwords

# =========================
# Настройки
# =========================
nltk.download("stopwords")
russian_stopwords = stopwords.words("russian")
ns = {"hl7": "urn:hl7-org:v3"}
input_dir = "epikriz2/epi"

# =========================
# Вспомогательные функции
# =========================
def clean_text(text):
    text = re.sub(r"<[^>]*>", " ", text)  # убрать теги
    text = re.sub(r"[^А-Яа-я\s]", " ", text)  # оставить только кириллицу
    text = text.lower()
    return " ".join([w for w in text.split() if w not in russian_stopwords])

def safe_float(value):
    try:
        clean_val = re.sub(r"[^0-9.]", "", value.replace(",", "."))
        return float(clean_val) if clean_val else None
    except:
        return None

def parse_xml_file(filepath):
    tree = ET.parse(filepath)
    root = tree.getroot()
    texts = []
    vitals = {}

    for section in root.findall(".//hl7:section", ns):
        title = section.find("hl7:title", ns)
        content = section.find("hl7:text", ns)
        if title is not None and content is not None:
            raw_text = ET.tostring(content, encoding="unicode")
            texts.append({
                "file": os.path.basename(filepath),
                "title": title.text.strip() if title.text else "",
                "text": raw_text,
                "cleaned": clean_text(raw_text)
            })

            txt = raw_text.lower()

            # Температура
            m = re.search(r"температура[^0-9]*([\d.,]+)", txt)
            if m:
                val = safe_float(m.group(1))
                if val is not None:
                    vitals["temperature"] = val

            # Давление
            m = re.search(r"(\d{2,3})[^\d]{1,3}(\d{2,3})\s*мм рт", txt)
            if m:
                s = safe_float(m.group(1))
                d = safe_float(m.group(2))
                if s is not None:
                    vitals["bp_systolic"] = s
                if d is not None:
                    vitals["bp_diastolic"] = d

            # Пульс
            m = re.search(r"пульс[^0-9]*([\d.,]+)", txt)
            if m:
                val = safe_float(m.group(1))
                if val is not None:
                    vitals["pulse"] = val

            # SpO2
            m = re.search(r"сатурац[^0-9]*([\d.,]+)", txt)
            if m:
                val = safe_float(m.group(1))
                if val is not None:
                    vitals["spo2"] = val

            # Рост
            m = re.search(r"рост[^0-9]*([\d]{2,3})", txt)
            if m:
                val = safe_float(m.group(1))
                if val is not None:
                    vitals["height"] = val

            # Вес
            m = re.search(r"вес[^0-9]*([\d]{2,3})", txt)
            if m:
                val = safe_float(m.group(1))
                if val is not None:
                    vitals["weight"] = val

    return texts, vitals

# =========================
# Обработка всех файлов
# =========================
all_texts = []
all_vitals = []

for fname in os.listdir(input_dir):
    if fname.endswith(".xml"):
        filepath = os.path.join(input_dir, fname)
        texts, vitals = parse_xml_file(filepath)
        all_texts.extend(texts)
        vitals["file"] = fname
        all_vitals.append(vitals)

df_texts = pd.DataFrame(all_texts)
df_vitals = pd.DataFrame(all_vitals).fillna(0)

print("Документов:", len(df_texts))
print("Пациентов с виталами:", len(df_vitals))
display(df_texts.head())
display(df_vitals.head())

# =========================
# Нормализация числовых данных
# =========================
scaler_minmax = MinMaxScaler()
scaler_std = StandardScaler()

df_vitals_minmax = scaler_minmax.fit_transform(df_vitals.drop(columns=["file"]))
df_vitals_std = scaler_std.fit_transform(df_vitals.drop(columns=["file"]))

print("MinMaxScaler пример:", df_vitals_minmax[:5])
print("StandardScaler пример:", df_vitals_std[:5])

# =========================
# Текст -> Число (TF-IDF)
# =========================
tfidf = TfidfVectorizer(max_features=500)
X_tfidf = tfidf.fit_transform(df_texts["cleaned"])

# =========================
# MLP (демонстрация)
# =========================
y = np.random.randint(0, 2, size=len(df_texts))

X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

mlp = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=300, random_state=42)
mlp.fit(X_train, y_train)

print("MLP Accuracy:", mlp.score(X_test, y_test))


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\PC2\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Документов: 26364
Пациентов с виталами: 2028


,file,title,text,cleaned
0,Эпикриз_100263_v1.xml,ДИАГНОЗЫ ПАЦИЕНТА,"<ns0:text xmlns:ns0=""urn:hl7-org:v3""> <ns...",заключительный клинический диагноз шифр тип те...
1,Эпикриз_100263_v1.xml,ЖАЛОБЫ ПРИ ПОСТУПЛЕНИИ,"<ns0:text xmlns:ns0=""urn:hl7-org:v3""> Сос...",состояние средней тяжести жалобы лихорадка тем...
2,Эпикриз_100263_v1.xml,СОЦИАЛЬНЫЙ АНАМНЕЗ,"<ns0:text xmlns:ns0=""urn:hl7-org:v3""> <ns...",занятость работающий место жительства город
3,Эпикриз_100263_v1.xml,Анамнез заболевания,"<ns0:text xmlns:ns0=""urn:hl7-org:v3""> В н...",ноябре г перенес коронавирусную инфекцию лабор...
4,Эпикриз_100263_v1.xml,ФИЗИКАЛЬНОЕ ОБСЛЕДОВАНИЕ,"<ns0:text xmlns:ns0=""urn:hl7-org:v3""> <ns...",средней степени тяжести ясное баллов обычной о...


,bp_systolic,bp_diastolic,pulse,height,weight,temperature,spo2,file
0,130.0,70.0,0.0,183.0,85.0,0.0,0.0,Эпикриз_100263_v1.xml
1,120.0,80.0,0.0,168.0,72.0,0.0,0.0,Эпикриз_100267_v1.xml
2,100.0,60.0,76.0,180.0,54.0,36.6,95.0,Эпикриз_100573_v1.xml
3,120.0,80.0,70.0,165.0,62.0,0.0,99.0,Эпикриз_10491_v1.xml
4,120.0,70.0,78.0,202.0,60.0,36.5,2.0,Эпикриз_10751_v1.xml


MinMaxScaler пример: [[0.59090909 0.5        0.         0.2267658  0.42079208 0.
  0.        ]
 [0.54545455 0.57142857 0.         0.20817844 0.35643564 0.
  0.        ]
 [0.45454545 0.42857143 0.54285714 0.22304833 0.26732673 0.92658228
  0.95959596]
 [0.54545455 0.57142857 0.5        0.20446097 0.30693069 0.
  1.        ]
 [0.54545455 0.5        0.55714286 0.25030979 0.2970297  0.92405063
  0.02020202]]
StandardScaler пример: [[ 1.39494794  1.10834401 -0.17409966  0.92141149  1.33506714 -0.07747413
  -0.11895162]
 [ 1.23053311  1.37287522 -0.17409966  0.74834281  1.02141237 -0.07747413
  -0.11895162]
 [ 0.90170345  0.84381281  4.79233249  0.88679776  0.58712117 12.89455807
   8.47393118]
 [ 1.23053311  1.37287522  4.40024574  0.71372908  0.78013948 -0.07747413
   8.83573677]
 [ 1.23053311  1.10834401  4.92302807  1.14063181  0.7318849  12.85911535
   0.06195118]]
MLP Accuracy: 0.5014223402237815
